In [ ]:


import os
import json
import pandas as pd
from datetime import datetime, timezone

from joblib import dump, load

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, brier_score_loss

from xgboost import XGBClassifier

import duckdb
DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"
con = duckdb.connect(DB_PATH) 
print("Connected:", DB_PATH)

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️ SHAP not installed — using permutation importance fallback.")


# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb


In [ ]:

# con.close()
# print("DuckDB connection closed.")

## Train DF

In [ ]:
train_df = con.execute('select* from prediction_data').df()
train_df = train_df[train_df["season_id"] != 20252026]
train_df = train_df.sort_values(by='game_id').reset_index(drop=True)


✅ streak_signed feature created successfully
Dropping 30 high-risk features...
Remaining feature count: 173


,season_id,game_id,game_date,team_id,tri_code,team_name,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,conference_team_count,conf_playoff_chance_baseline,streak_signed,rank_bucket
0,20132014,2013020006,2013-10-02,24,ANA,Anaheim Ducks,0,<NA>,0,NaN,NaN,0.494599,2,2.0,240.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,4,4.0,0.500000,0.50000,-2,-2.0,369.0,2,2.0,0,0.0,2.0,0.0,0.0,6,2.0,4.0,50.0,6,0.79310,6.00000,2,2,0,2,2,0,2.0,2.000000,0.000000,59,58,1,44.0,42.0,2.0,1,1,59.000000,58.000000,1.000000,23,36,1,51.0,10,19,31,8,16,32,0.555556,0.444444,0.542857,0.457143,0.492063,0.507937,23,36,1,51.0,23.000000,51.000000,10,8,19,16,31,32,36,8,8,21,1,0,0,2.777778,29,4,9,16,79.310345,82.088123,1,8,8,21,1,0,0,6,4,9,16,0.027778,0.793103,82.088123,59,58,44.0,42.0,2.0,2.0,81,0,0.000000,1,0,0,0.000000,0,0,0,0,0.000000,0,0,1.000000,-5,-5.00,1.0,1,0,1,0,0,1,6,-5,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,6,-5,Pacific,7,1,7,7,Western,14,3,14,14,30,4,30,30,8,1,1,6,0,0,0.00,0.000000,4.758600,0,2.00000,14,0.571429,-1,tier4_bottom
1,20132014,2013020029,2013-10-05,24,ANA,Anaheim Ducks,1,0,1,0.000000,0.000000,0.494599,3,3.0,348.0,1,1.0,1,1.0,0.33333,0.333333,0,0.0,4,4.0,0.500000,0.50000,-2,-2.0,303.0,2,2.0,0,0.0,5.0,1.0,20.0,9,4.0,8.0,50.0,15,0.90909,2.77278,4,4,0,6,6,0,3.0,3.000000,0.000000,43,42,1,31.0,30.0,1.0,2,2,51.000000,50.000000,1.000000,28,30,4,55.0,7,15,22,9,19,27,0.437500,0.562500,0.441176,0.558824,0.448980,0.551020,51,66,5,106.0,25.500000,53.000000,17,17,34,35,53,59,30,5,10,15,3,1,0,13.333333,33,10,8,15,90.909091,104.242424,5,13,18,36,4,1,0,9,14,17,31,0.075758,0.854839,93.059629,102,100,75.0,72.0,3.0,1.5,80,1,0.500000,1,0,1,0.500000,0,0,0,2,0.500000,82,0,2.500000,-4,-2.00,1.0,2,1,1,0,2,5,9,-4,1,0,0,0,0,0,0,0,0,0,0,0,2,1,1,0,1,0,2,5,9,-4,Pacific,5,4,2,5,Western,10,10,3,10,17,19,8,17,4,1,4,3,1,1,0.50,0.247299,2.520707,0,2.00000,14,0.571429,1,tier3_bubble
2,20132014,20

In [ ]:


assert (
    train_df
    .groupby(["season_id", "tri_code"])["made_playoffs"]
    .nunique()
    .le(1)
    .all()
), "made_playoffs varies within a team-season"
train_df = train_df.sort_values(by=['season_id','tri_code', 'game_id'])

conference_sizes = (
    train_df.groupby(["season_id", "conference"])["tri_code"]
    .nunique()
    .reset_index()
    .rename(columns={"tri_code": "conference_team_count"})
)

train_df = train_df.merge(conference_sizes, on=["season_id", "conference"], how="left")

train_df["conf_playoff_chance_baseline"] = 8 / train_df["conference_team_count"]

train_df["streak_direction"] = train_df["streak"].map({
    "W": 1,
    "L": -1
}).fillna(0).astype(int)


train_df["streak_signed"] = (
    train_df["streak_direction"] *
    train_df["streak_count"].fillna(0).astype(int)
)


train_df["streak_signed"] = train_df["streak_signed"].clip(-7, 7)


train_df.drop(columns=["streak", "streak_direction"], inplace=True)


assert (
    train_df
    .groupby(["season_id", "tri_code", "game_date"])["streak_signed"]
    .nunique()
    .le(1)
    .all()
), "❌ streak_signed varies within a team-day"

print("✅ streak_signed feature created successfully")
def bucket_rank(rank):
    if rank <= 3:
        return "tier1_elite"
    elif rank <= 6:
        return "tier2_strong"
    elif rank <= 10:
        return "tier3_bubble"
    else:
        return "tier4_bottom"
        
train_df["rank_bucket"] = train_df["conference_rank"].apply(bucket_rank)

# =====================================================
# DROP HIGH-NOISE / HIGH-RISK FEATURES (MODEL ONLY)
# =====================================================

DROP_COLS = [

    # -------------------------------------------------
    # 1. SINGLE-GAME RAW STATS (NOISY)
    # -------------------------------------------------
    "shots_for",
    "shots_against",
    "goals_for",
    "goals_against",

    "hd_sog",
    "md_sog",
    "ld_sog",

    "hd_goals",
    "md_goals",
    "ld_goals",

    "hd_against",
    "md_against",
    "ld_against",

    "game_savePct",
    "game_GAA",

    # -------------------------------------------------
    # 2. RAW PER-GAME SPECIAL TEAMS COUNTS
    # (keep cumulative / per-game rates instead)
    # -------------------------------------------------
    "pp_gf",
    "pp_ga",
    "sh_gf",
    "sh_ga",
    "times_sh",

    # -------------------------------------------------
    # 3. REDUNDANT / DOMINANT RANK FEATURES
    # (keep rank_bucket + conference_rank only)
    # -------------------------------------------------

    "division_home_rank",
    "division_road_rank",
    "division_last_10_rank",

    "conference_home_rank",
    "conference_road_rank",
    "conference_last_10_rank",

    "league_rank",
    "league_home_rank",
    "league_road_rank",
    "league_last_10_rank",



    # -------------------------------------------------
    # 4. OPTIONAL: STRONG LATE-SEASON PROXIES
    # (uncomment if probabilities converge too fast)
    # -------------------------------------------------
    # "in_playoff_spot",
]

# =====================================================
# APPLY DROP (SAFE — ignores missing)
# =====================================================

print(f"Dropping {len(DROP_COLS)} high-risk features...")

train_df_model = train_df.drop(columns=DROP_COLS, errors="ignore")

print("Remaining feature count:", train_df_model.shape[1])

train_df.head()



In [10]:
test_df = con.execute('select* from current_data').df()
test_df = test_df.sort_values(by=['season_id','tri_code', 'game_id'])

conference_sizes = (
    test_df.groupby(["season_id", "conference"])["tri_code"]
    .nunique()
    .reset_index()
    .rename(columns={"tri_code": "conference_team_count"})
)

test_df = test_df.merge(conference_sizes, on=["season_id", "conference"], how="left")

test_df["conf_playoff_chance_baseline"] = 8 / test_df["conference_team_count"]
test_df["streak_direction"] = test_df["streak"].map({
    "W": 1,
    "L": -1
}).fillna(0).astype(int)


test_df["streak_signed"] = (
    test_df["streak_direction"] *
    test_df["streak_count"].fillna(0).astype(int)
)


test_df["streak_signed"] = test_df["streak_signed"].clip(-7, 7)


test_df.drop(columns=["streak", "streak_direction"], inplace=True)


assert (
    test_df
    .groupby(["season_id", "tri_code", "game_date"])["streak_signed"]
    .nunique()
    .le(1)
    .all()
), "❌ streak_signed varies within a team-day"

print("✅ streak_signed feature created successfully")
def bucket_rank(rank):
    if rank <= 3:
        return "tier1_elite"
    elif rank <= 6:
        return "tier2_strong"
    elif rank <= 10:
        return "tier3_bubble"
    else:
        return "tier4_bottom"
        
test_df["rank_bucket"] = test_df["conference_rank"].apply(bucket_rank)
# =====================================================
# DROP HIGH-NOISE / HIGH-RISK FEATURES (MODEL ONLY)
# =====================================================

DROP_COLS = [

    # -------------------------------------------------
    # 1. SINGLE-GAME RAW STATS (NOISY)
    # -------------------------------------------------
    "shots_for",
    "shots_against",
    "goals_for",
    "goals_against",

    "hd_sog",
    "md_sog",
    "ld_sog",

    "hd_goals",
    "md_goals",
    "ld_goals",

    "hd_against",
    "md_against",
    "ld_against",

    "game_savePct",
    "game_GAA",

    # -------------------------------------------------
    # 2. RAW PER-GAME SPECIAL TEAMS COUNTS
    # (keep cumulative / per-game rates instead)
    # -------------------------------------------------
    "pp_gf",
    "pp_ga",
    "sh_gf",
    "sh_ga",
    "times_sh",

    # -------------------------------------------------
    # 3. REDUNDANT / DOMINANT RANK FEATURES
    # (keep rank_bucket + conference_rank only)
    # -------------------------------------------------

    "division_home_rank",
    "division_road_rank",
    "division_last_10_rank",

    "conference_home_rank",
    "conference_road_rank",
    "conference_last_10_rank",

    "league_rank",
    "league_home_rank",
    "league_road_rank",
    "league_last_10_rank",



    # -------------------------------------------------
    # 4. OPTIONAL: STRONG LATE-SEASON PROXIES
    # (uncomment if probabilities converge too fast)
    # -------------------------------------------------
    # "in_playoff_spot",
]

# =====================================================
# APPLY DROP (SAFE — ignores missing)
# =====================================================

print(f"Dropping {len(DROP_COLS)} high-risk features...")

current_df = test_df.drop(columns=DROP_COLS, errors="ignore")

print("Remaining feature count:", current_df.shape[1])

current_df.head()


✅ streak_signed feature created successfully
Dropping 30 high-risk features...
Remaining feature count: 172


,season_id,game_id,game_date,team_id,tri_code,team_name,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga_pg,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga_pg,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shot_pct,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,conference,conference_rank,wildcard_rank,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,conference_team_count,conf_playoff_chance_baseline,streak_signed,rank_bucket
0,20252026,2025020021,2025-10-09,24,ANA,Anaheim Ducks,0,<NA>,0,NaN,NaN,0.495047,1,1.0,120.0,0.0,0,0.0,0.00000,0.000000,0.0,2.0,1.000000,1.00000,0,0.0,240.0,0.0,0.0,1.0,0.0,0.0,9,0.0,2.0,0.0,1,17,11,6,17,11,6,17.0,11.000000,6.000000,87,86,1,57,60,-3,1,1,87.000000,86.000000,1.00,24,36,1,60,19,17,52,9,15,41,0.678571,0.321429,0.531250,0.468750,0.559140,0.440860,24,36,1,60,24.000000,60.000000,19,9,17,15,52,41,2.777778,90.322581,93.100358,1,6,9,21,1,0,0,3,5,9,17,0.027778,0.903226,93.100358,87,86,57,60,-3,-3.000000,81,0,0.000000,1,0,0,0.000000,0,0,0,0,0.000000,0,0,1.00,-2,-2.00,1.0,1,0,1,0,0,1,3,-2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,3,-2,Pacific,8,Western,15,9,0,0,0.00,0.000000,2.851438,-12,2.00000,16,0.5,-1,tier4_bottom
1,20252026,2025020036,2025-10-11,24,ANA,Anaheim Ducks,1,0,1,0.000000,0.000000,0.495047,4,4.0,428.0,2.0,2,2.0,0.50000,0.500000,0.0,4.0,0.500000,0.50000,-2,-2.0,343.0,2.0,0.0,5.0,2.0,40.0,3,2.0,6.0,33.3,12,8,9,-1,25,20,5,12.5,10.000000,2.500000,85,84,1,64,58,6,2,2,86.000000,85.000000,1.00,43,44,7,75,32,19,38,28,19,36,0.533333,0.466667,0.500000,0.500000,0.513514,0.486486,67,80,8,135,33.500000,67.500000,51,37,36,34,90,77,15.909091,73.913043,89.822134,8,22,20,41,6,3,0,9,9,19,26,0.100000,0.833333,93.333333,172,170,121,118,3,1.500000,80,1,0.500000,1,0,1,0.500000,0,0,0,2,0.500000,82,0,4.00,-1,-0.50,1.0,2,1,1,0,2,8,9,-1,1,0,0,0,0,0,0,0,0,0,0,0,2,1,1,0,1,0,2,8,9,-1,Pacific,5,Western,12,6,1,1,0.50,0.247524,4.448365,1,2.00000,16,0.5,1,tier4_bottom
2,20252026,2025020056,2025-10-14,24,ANA,Anaheim Ducks,1,1,2,0.500000,0.333333,0.512727,6,6.0,429.0,2.0,2,2.0,0.33333,0.333333,0.0,1.0,1.000000,1.00000,0,0.0,120.0,0.0,0.0,11.0,4.0,36.4,2,2.0,7.0,28.6,14,11,9,2,36,29,7,12.0,9.666667,2.333333,69,68,1,43,44,-1,3,3,80.333333,79.333333,1.00,37,22,4,73,13,29,29,15,26,23,0.464286,0.535714,0.527273,0.472727,0.557692,0.442308,104,102,12,208,34.666667,69.333333,64,52,65,60,119,100,18.181818,88.000000,106.181818,12,25,29,51,7,4,2,12,20,26,34,0.117647,0.848101,96.574832,241,238,164,162,2,0.666667,79,2,0.666667,1,0,2,0.666667,1,0,0,4,0.666667,109,0,4.00,0,0.00

In [ ]:
# =====================================================
# PLAYOFF MODEL — WC FIELD PRESSURE v1 (LOCKED)
# =====================================================
# Do NOT modify without bumping version.
# Brier ~ 0.058
# =====================================================

# =====================================================
# 1. CONSTANTS
# =====================================================

MODEL_VERSION = "wc_field_pressure_v1"

MODEL_PATH = f"models/playoff_xgb_pipeline_{MODEL_VERSION}.joblib"
FEATURE_PATH = f"models/playoff_features_{MODEL_VERSION}.json"

LEAK_COLS = [
    "game_id", "game_date", "date",
    "clinch_type", "made_playoffs",
    "team_name", "opponent",
    "season_id"
]

ID_COLS = [
    "team_id", "opponent_id", "tri_code"
]

# =====================================================
# 2. LOCKED FEATURE ENGINEERING
# =====================================================

def add_wc_field_pressure_features(df):
    """
    Add NHL wild-card cutline + pressure features.
    LOCKED for v1.
    """

    out = df.copy()

    wc_eligible = out[out["division_rank"] > 3]

    wc_cutline = (
        wc_eligible
        .sort_values(
            ["season_id", "game_date", "conference", "points", "points_pct"],
            ascending=[True, True, True, False, False]
        )
        .groupby(["season_id", "game_date", "conference"], as_index=False)
        .nth(1)
        .reset_index(drop=True)
        [["season_id", "game_date", "conference", "points", "points_pct"]]
        .rename(columns={
            "points": "wc2_points",
            "points_pct": "wc2_points_pct"
        })
    )

    out = out.merge(
        wc_cutline,
        on=["season_id", "game_date", "conference"],
        how="left"
    )

    out["wc_points_margin"] = out["points"] - out["wc2_points"]
    out["wc_points_pct_margin"] = out["points_pct"] - out["wc2_points_pct"]

    out["in_wc_spot"] = (out["wildcard_rank"] <= 2).astype(int)
    out["in_div_top3"] = (out["division_rank"] <= 3).astype(int)

    out["on_wc_bubble"] = out["wildcard_rank"].between(1, 4).astype(int)

    return out


def add_wc_density_features(df, windows=(2, 4, 6)):
    """
    Add WC crowding / density features.
    LOCKED for v1.
    """

    out = df.copy()
    wc_mask = out["division_rank"] > 3

    for w in windows:
        col = f"wc_teams_within_{w}pts"

        out[col] = (
            out
            .assign(diff=lambda x: (x["points"] - x["wc2_points"]).abs())
            .where(wc_mask)
            .groupby(["season_id", "game_date", "conference"])["diff"]
            .transform(lambda s: (s <= w).sum())
            .fillna(0)
            .astype(int)
        )

    return out

# =====================================================
# 3. TRAIN MODEL
# =====================================================

def train_playoff_model(train_df):
    """
    Train WC field-pressure playoff model.
    """

    df = (
        train_df
        .sort_values(["season_id", "team_id", "game_date"])
        .reset_index(drop=True)
    )

    df = add_wc_field_pressure_features(df)
    df = add_wc_density_features(df)

    X = (
        df
        .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
        .select_dtypes(include="number")
    )

    X = X.loc[:, X.isna().mean() < 0.20].fillna(0)
    y = df["made_playoffs"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", XGBClassifier(
                n_estimators=500,
                max_depth=5,
                learning_rate=0.01,
                subsample=0.8,
                colsample_bytree=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=42,
                n_jobs=-1,
            )),
        ]
    )

    pipeline.fit(X_train, y_train)

    y_prob = pipeline.predict_proba(X_test)[:, 1]

    print("\n✅ MODEL REPORT\n")
    print(classification_report(y_test, pipeline.predict(X_test)))
    print("Brier score:", brier_score_loss(y_test, y_prob))

    return pipeline, X.columns.tolist()

# =====================================================
# 4. SAVE LOCKED ARTIFACTS
# =====================================================

def save_model_bundle(pipeline, feature_cols):
    os.makedirs("models", exist_ok=True)

    dump(
        {
            "pipeline": pipeline,
            "features": feature_cols,
            "version": MODEL_VERSION,
            "trained_on": datetime.now(timezone.utc).isoformat(),
            "notes": "WC cutline + density features; race-aware calibration",
        },
        MODEL_PATH
    )

    with open(FEATURE_PATH, "w") as f:
        json.dump(feature_cols, f, indent=2)

    print(f"\n💾 Model saved → {MODEL_PATH}")
    print(f"📐 Features saved → {FEATURE_PATH}")

# =====================================================
# 5. LOAD + SCORE (INFERENCE ENTRY POINT)
# =====================================================

def load_and_score(current_df):
    """
    Canonical inference path for v1 model.
    """

    bundle = load(MODEL_PATH)
    pipeline = bundle["pipeline"]
    feature_cols = bundle["features"]

    df = add_wc_field_pressure_features(current_df)
    df = add_wc_density_features(df)

    X = (
        df
        .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
        .select_dtypes(include="number")
        .fillna(0)
        .reindex(columns=feature_cols, fill_value=0)
    )

    df["playoff_prob_row"] = pipeline.predict_proba(X)[:, 1]

    return df

# =====================================================
# END OF FILE — LOCKED v1
# =====================================================


In [ ]:
# pipeline, feature_cols = train_playoff_model(train_df_model)
# save_model_bundle(pipeline, feature_cols)






✅ MODEL REPORT

              precision    recall  f1-score   support

         0.0       0.92      0.92      0.92      2150
         1.0       0.92      0.93      0.92      2308

    accuracy                           0.92      4458
   macro avg       0.92      0.92      0.92      4458
weighted avg       0.92      0.92      0.92      4458

Brier score: 0.05894000325360196

💾 Model saved → models/playoff_xgb_pipeline_wc_field_pressure_v1.joblib
📐 Features saved → models/playoff_features_wc_field_pressure_v1.json


C:\Users\kjchr\AppData\Local\Temp\ipykernel_19404\1061688889.py:173: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "trained_on": datetime.utcnow().isoformat(),


In [ ]:
from playoff_model_wc_pressure_v1 import load_and_score

current_df = load_and_score(current_df)


ModuleNotFoundError: No module named 'playoff_model_wc_pressure_v1'